<a href="https://colab.research.google.com/github/hamzafareed123/code-review-agent/blob/fetch-pr-node/code_review_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB-TOKEN")


In [ ]:
!pip install PyGithub langgraph langchain langchain-core langchain-community langchain-groq

In [13]:
from github import Github, Auth
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Literal
from pydantic import BaseModel,Field

In [43]:
#==================================
#  Graph STATE
#==================================

class PRState(TypedDict):
  repo_name:str
  pr_title:str
  pr_body:str
  pr_number:int
  pr_state:Literal["open","closed","merged"]
  files:list[dict]

In [44]:
#==================================
#  Fetch Pull Request NODE
#==================================

def fetch_pr_node(state:PRState):
  auth = Auth.Token(GITHUB_TOKEN)
  g= Github(auth=auth)
  repo = g.get_repo(state["repo_name"])
  pr= repo.get_pull(state['pr_number'])

  files = pr.get_files()

  for f in files:
    state["files"].append({
        "filename": f.filename,
        "additions": f.additions,
        "deletions": f.deletions,
        "changes": f.changes,
        "patch": f.patch  })

  pr_body_content = pr.body if pr.body else "No Description Provided"
  return {"pr_title":pr.title,"pr_body":pr_body_content,"pr_number":pr.number,"pr_state":pr.state}

In [45]:
#==================================
#  Graph Build STATE
#==================================

graph = StateGraph(PRState)

graph.add_node("fetch_pr_node",fetch_pr_node)

graph.add_edge(START,"fetch_pr_node")
graph.add_edge("fetch_pr_node",END)

workflow = graph.compile()

In [ ]:
workflow

In [47]:
initial_state = {"repo_name":"hamzafareed123/async-board","pr_title":"","pr_body":"","pr_number":49,"pr_state":"open","files":[]}

result=workflow.invoke(initial_state)


In [ ]:
print(result)